# 22 — Train-only EDA

**Objetivo:** estudiar distribución, missingness y cardinalidad usando exclusivamente TRAIN. Validation y Test no participan en decisiones exploratorias de features.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.train_eda import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

In [ ]:
from argos.gold.train_eda import build_train_eda

result = build_train_eda(
    silver_dir=paths.silver,
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
)

In [ ]:
display(result.target_profile)
display(result.feature_profile)
display(result.category_cardinality)
display(result.numeric_summary)

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("22 ✅ PASS" if result.ready_for_next_notebook else "22 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook, "Train-only EDA no supera sus checks bloqueantes."